In [ ]:
import pandas as pd
import torch
import numpy as np
import pandas as pd
import random
from sklearn import preprocessing
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt

## Setup synset evaluation

In [ ]:
def run_evaluation(sense_inventory):
    # GENERATE SYNSETS
    print("Generating synsets...")
    df_synset = generate_synsets(sense_inventory)
    
    # LOAD BORRA WORDNET
    print("Reading Borra Wordnet...")
    df_fwn = pd.read_csv(fr"D:\thesis\filwordnet\wordnet.csv") 
    
    # EVALUATE SYNSETS
    print("Evaluating synsets...")
    df_results = evaluate_synsets(df_synset, df_fwn)
    
    return df_results
    
def generate_synsets(sense_inventory):
    '''
        df_sense_embeddings: columns=['word', 'sense_id', 'example_sentences', 'pos', 'contextual_info', 'sense_embedding']
    '''
    df_synsets = sense_inventory.copy()
    sense_embeddings = list(sense_inventory['sense_embedding'])
    
    distance_threshold = 0.12
    af = AgglomerativeClustering(n_clusters=None, 
                                     affinity="cosine", 
                                     linkage="average", 
                                     distance_threshold=distance_threshold).fit(sense_embeddings)

    df_synsets['synset_id'] = af.labels_

    return df_synsets

def get_jaccard_index(lst_a, lst_b):
    intersection = list(set(lst_a).intersection(lst_b))
    union = list(set(lst_a).union(lst_b))
    return len(intersection)/len(union)
    
def evaluate_synsets(df_synset, df_fwn):
    '''
    DATAFRAME VERSION
    Returns DataFrame with the ff. columns: our_synset_index, fwn_synset_index, our_synsets, fwn_synsets, jaccard_scores
    Does not evaluate synsets with single item.
    '''
    
    ##############################
    # PREPROCESSING
    ##############################
    # Preprocess FilWordNet synsets for evaluation
    fwn_synsets = []
    fwn_synsets_with_definition = []
    for synsetid in df_fwn['synsetid'].unique():
        synset_item = list(df_fwn[df_fwn['synsetid'] == synsetid]['lemma'])
        fwn_synsets.append(synset_item)
        synset_item_definition =  list(df_fwn[df_fwn['synsetid'] == synsetid]['definition']) 
        fwn_synsets_with_definition.append(synset_item_definition)

    # Preprocess our synsets for evaluation
    our_synsets = []
    our_synsets_with_sense_id = []
    for synset_id in df_synset['synset_id'].unique():
        synset_item = list(df_synset[df_synset['synset_id'] == synset_id]['word'])
        our_synsets.append(synset_item)
        synset_item_sense_id = list(df_synset[df_synset['synset_id'] == synset_id]['sense_id'])
        our_synsets_with_sense_id.append(synset_item_sense_id)
        
    ##############################
    # BUILD JACCARD INDEX MATRIX
    ##############################
    jaccard_matrix = [] # store 
    invalid_synset_idx = []
    for idx, our_synset in enumerate(our_synsets):
        if len(our_synset) == 1:
            invalid_synset_idx.append(idx)

        score = []
        for fwn_synset in fwn_synsets:
            if len(fwn_synset) == 1: # if fwn synset is just 1, make the score -1 para never ma select sa argmax
                score.append(-1)
            else:
                score.append(get_jaccard_index(our_synset, fwn_synset))
        
        jaccard_matrix.append(score)

    ##############################
    # GET HIGHEST JACCARD INDEX
    ##############################
    jaccard_matrix = np.array(jaccard_matrix)

    ##############################
    # BUILD RESULTS (our_synset_index, fwn_synset_index, jaccard_scores)
    # REMOVE INVALID SYNSETS (SINGLE ITEM SYNSETS)
    ##############################
    our_synset_index = np.array(range(len(our_synsets)))
    our_synset_index = np.delete(our_synset_index, invalid_synset_idx)
    fwn_synset_index = np.argmax(jaccard_matrix, axis=1)
    fwn_synset_index = np.delete(fwn_synset_index, invalid_synset_idx)
    
    our_synsets = np.array(our_synsets_with_sense_id)[our_synset_index] # uncomment if gusto may sense_id sa our_synsets e.g. pangit_0, pangit_1

    
    fwn_synsets = np.array(fwn_synsets)[fwn_synset_index]
    fwn_synsets_with_definition = np.array(fwn_synsets_with_definition)[fwn_synset_index]

    jaccard_scores = np.round(jaccard_matrix[our_synset_index, fwn_synset_index], decimals=2)

    df_results = pd.DataFrame(data={'our_synset_index': our_synset_index, 'fwn_synset_index': fwn_synset_index, 'our_synsets': our_synsets, 'fwn_synsets': fwn_synsets, 'fwn_definition': fwn_synsets_with_definition, 'jaccard_scores': jaccard_scores}).sort_values(by=['jaccard_scores'], ascending=False).reset_index(drop=True)

    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', 150):
        display(df_results[['our_synsets', 'fwn_synsets', 'fwn_definition', 'jaccard_scores']])

    return df_results

## Load wordnet

In [ ]:
filwordnet = pd.read_pickle(r"..\output\filwordnet.pkl")
filwordnet

In [ ]:
results = run_evaluation(filwordnet)